# iter8ml — Telco Churn in one call

Run the full iter8ml loop — a cross-validated CatBoost-vs-XGBoost **leaderboard** plus a **SHAP** explanation of the champion — on the bundled Telco Churn sample. This is the interactive counterpart to the [static demo page](https://minghao51.github.io/iter8ml/notebooks/demo-telco-churn/).

Repo: https://github.com/minghao51/iter8ml

In [ ]:
# Install from git so the in-package bundled dataset is available.
!pip install -q "iter8ml[gbdt] @ git+https://github.com/minghao51/iter8ml"

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from iter8ml import ExperimentConfig, ExperimentSession, TaskType, load_data
from iter8ml.config import HardwareProfile
from iter8ml.constants import CVStrategy
from iter8ml.data.adapter import DataAdapter
from iter8ml.datasets import bundled_dataset_path
from iter8ml.engine.models.factory import get_model_class
from iter8ml.workspace import Workspace

# Cap OpenMP threads before any GBDT model is instantiated. GBDT libs load
# lazily on get_model_class() below; this avoids libgomp deadlock on
# hybrid-core P+E CPUs under Linux/WSL2.
HardwareProfile.configure_omp_threads()

TARGET = "Churn"
DATA = str(bundled_dataset_path("telco_churn"))
df = load_data(DATA)
print(f"{len(df):,} rows x {df.width - 1} features")
df.head()

In [ ]:
# Same core as demo/app.py::run_analysis — catboost + xgboost, 5-fold
# stratified CV, roc_auc + f1_macro, in an isolated throwaway workspace.
METRICS = ["roc_auc", "f1_macro"]
MODELS = ["catboost", "xgboost"]

with tempfile.TemporaryDirectory() as tmp:
    ws = Workspace(root=Path(tmp))
    ws.init()
    session = ExperimentSession(workspace=ws, tracker=None)
    config = ExperimentConfig(
        name="demo",
        task=TaskType.CLASSIFICATION,
        target_col=TARGET,
        data_path="",
        models=MODELS,
        cv_folds=5,
        cv_strategy=CVStrategy.STRATIFIED,
        metrics=METRICS,
        shap_enabled=False,
        random_seed=42,
        max_workers=1,
    )
    session.run(config, df)
    lb = session.leaderboard()

lb.select("model", "primary_metric", "primary_score", "duration_seconds").sort(
    "primary_score", descending=True
)

In [ ]:
import contextlib

import shap
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# SHAP beeswarm on the champion. GBDTs need numeric input, so ordinal-encode
# object columns first (mirrors benchmarks/openml_benchmark.py). CatBoost
# handles categoricals natively during CV; this encoded refit is just for
# the SHAP TreeExplainer.
X, y = DataAdapter().transform(df, TARGET)
for i in range(X.shape[1]):
    if X[:, i].dtype == object:
        with contextlib.suppress(ValueError, TypeError):
            X[:, i] = X[:, i].astype(float)
str_cols = [i for i in range(X.shape[1]) if X[:, i].dtype == object]
if str_cols:
    X[:, str_cols] = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1
    ).fit_transform(X[:, str_cols])
X = X.astype(np.float64)
y = LabelEncoder().fit_transform(np.asarray(y).astype(str))

champion = get_model_class("catboost")(task="classification")
champion.fit(X, y)

sv = shap.TreeExplainer(champion._model)(X[:500])
shap.plots.beeswarm(sv, max_display=12, show=False)
feat = [c for c in df.columns if c != TARGET]
top_driver = feat[int(np.argsort(np.abs(sv.values).mean(0))[-1])]
plt.title(f"Top churn driver: {top_driver}")
plt.tight_layout()
plt.show()